In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import preprocessing as pp, feature_engineering as fe, pipeline as P
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
def show(name, w=800): display(Image(filename=str(FIG / name), width=w))

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 06 Feature engineering
Only facility-characteristic and volume features; **no index measure** is used as a feature (leakage control).

In [2]:
m = pd.read_csv(PROC/'irf_facilities_merged.csv', dtype={CCN:str}, parse_dates=['Certification Date'])
X = fe.engineer_features(m); fe.FEATURE_DICTIONARY

,feature,formula / source,type,purpose / rationale
0,ownership,Ownership Type (General file),categorical,Ownership differs in incentives/structure; des...
1,cms_region,CMS Region 1-10,categorical,Geography/administrative region.
2,facility_age_years,2026 - year(Certification Date),numeric,Years in the Medicare programme; experience pr...
3,n_irf_in_state,count of IRFs in the facility's state,numeric,Local IRF market density (computed from Genera...
4,name_has_rehab,1 if 'REHAB' in Provider Name,binary,Heuristic for free-standing rehabilitation hos...
5,name_encompass_chain,1 if 'ENCOMPASS' in Provider Name,binary,Heuristic flag for the largest national IRF op...
6,log_eligible_stays,ln(1 + I_017_01_PPR_PD_VOLUME),numeric,Facility size/volume (eligible stays); log red...
7,log_mspb_episodes,ln(1 + I_020_01_MSPB_NUMB),numeric,Volume of eligible Medicare episodes.
8,log_function_cases,ln(1 + I_026_01_DENOMINATOR),numeric,Volume of assessed patients.
9,I_016_01_OBS_RATE (set B only),HCP flu-vaccination rate,numeric,"Held-out quality measure, NOT in the index (ab..."


In [3]:
X.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ownership,1222,4,Non-profit,569,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cms_region,1222,10,6,273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_age_years,1222.0,NaN,NaN,NaN,21.026187,13.992544,0.0,7.25,21.0,34.0,43.0
n_irf_in_state,1222.0,NaN,NaN,NaN,54.880524,45.875205,1.0,23.0,41.0,83.0,157.0
name_has_rehab,1222.0,NaN,NaN,NaN,0.456628,0.498319,0.0,0.0,0.0,1.0,1.0
name_encompass_chain,1222.0,NaN,NaN,NaN,0.104746,0.306352,0.0,0.0,0.0,0.0,1.0
log_eligible_stays,1135.0,NaN,NaN,NaN,5.943515,0.978115,1.386294,5.327876,5.934894,6.659294,8.523374
log_mspb_episodes,1106.0,NaN,NaN,NaN,5.930534,1.009705,0.693147,5.313206,5.880533,6.696416,8.591373
log_function_cases,1209.0,NaN,NaN,NaN,5.532943,1.629917,0.0,5.129899,5.780744,6.51323,8.375169


In [4]:
X.isna().mean().round(3)

ownership               0.000
cms_region              0.000
facility_age_years      0.000
n_irf_in_state          0.000
name_has_rehab          0.000
name_encompass_chain    0.000
log_eligible_stays      0.071
log_mspb_episodes       0.095
log_function_cases      0.011
dtype: float64